In [5]:
import re
import json
import pandas as pd
import os
from pathlib import Path

In [6]:
os.chdir("..")

In [7]:
date = "27APR2026"

In [8]:


def extract_snapshots(log_file_path):
    pattern = re.compile(r'WINDOW_SNAPSHOT:\s*(\{.*\})')

    snapshots = []

    with open(log_file_path, 'r') as f:
        for line in f:
            match = pattern.search(line)
            if match:
                try:
                    data = json.loads(match.group(1))
                    snapshots.append(data)
                except json.JSONDecodeError:
                    continue

    return snapshots


def snapshots_to_dataframe(snapshots):
    df = pd.DataFrame(snapshots)

    # Optional: convert time column
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])

    return df


In [9]:
# ---- Usage ----
log_path = Path(f"assets/logs/{date}/streamer.log")


snapshots = extract_snapshots(log_path)
df = snapshots_to_dataframe(snapshots)

In [10]:
df.shape

(656, 17)

In [11]:
df.head()

,time,nifty,atm,ATM-3_CE,ATM-3_PE,ATM-2_CE,ATM-2_PE,ATM-1_CE,ATM-1_PE,ATM_CE,ATM_PE,ATM+1_CE,ATM+1_PE,ATM+2_CE,ATM+2_PE,ATM+3_CE,ATM+3_PE
0,2026-04-27 09:10:05.785,23945.45,23950,NIFTY26APR23800CE,NIFTY26APR23800PE,NIFTY26APR23850CE,NIFTY26APR23850PE,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE
1,2026-04-27 09:15:01.784,23980.80,24000,NIFTY26APR23850CE,NIFTY26APR23850PE,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE
2,2026-04-27 09:15:08.281,24027.85,24050,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE
3,2026-04-27 09:15:08.781,24013.75,24000,NIFTY26APR23850CE,NIFTY26APR23850PE,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE
4,2026-04-27 09:15:09.281,24032.80,24050,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE


In [12]:
# df.to_excel("option_chain.xlsx")

In [13]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill
import random

# --- Config ---
COLOR_POOL = [
    "FFC7CE", "C6EFCE", "FFEB9C", "BDD7EE", "D9D2E9",
    "FCE4D6", "E2EFDA", "FFF2CC", "DDEBF7", "EAD1DC",

    "F4CCCC", "D9EAD3", "FFF2CC", "CFE2F3", "D9D2E9",
    "FCE5CD", "EAD1DC", "D0E0E3", "F9CB9C", "C9DAF8",

    "EA9999", "B6D7A8", "FFE599", "9FC5E8", "B4A7D6",
    "F6B26B", "D5A6BD", "A2C4C9", "FFD966", "A4C2F4",

    "E06666", "93C47D", "FFD966", "6FA8DC", "8E7CC3",
    "F6B26B", "C27BA0", "76A5AF", "F1C232", "6D9EEB",

    "CC0000", "6AA84F", "F1C232", "3D85C6", "674EA7",
    "E69138", "A64D79", "45818E", "BF9000", "3C78D8",

    "990000", "38761D", "BF9000", "134F5C", "351C75",
    "783F04"
]

# --- Persistent mapping ---
symbol_color_map = {}

def get_color(symbol):
    if symbol not in symbol_color_map:
        # deterministic or random
        color = COLOR_POOL[len(symbol_color_map) % len(COLOR_POOL)]
        symbol_color_map[symbol] = color
    return symbol_color_map[symbol]


def write_colored_excel(df, file_name="output.xlsx"):
    wb = Workbook()
    ws = wb.active

    # Write header
    ws.append(list(df.columns))

    for row_idx, row in df.iterrows():
        excel_row = []

        for col in df.columns:
            value = row[col]
            excel_row.append(value)

        ws.append(excel_row)

        # Apply colors AFTER writing row
        for col_idx, col in enumerate(df.columns, start=1):
            value = row[col]

            if isinstance(value, str) and value.startswith("NIFTY"):
                color = get_color(value)

                fill = PatternFill(
                    start_color=color,
                    end_color=color,
                    fill_type="solid"
                )

                ws.cell(row=row_idx + 2, column=col_idx).fill = fill

    wb.save(file_name)

In [14]:
path_ = Path(f"assets/logs/{date}/extracted_symbols/option_chain_colored.xlsx")
path_.parent.mkdir(parents=True, exist_ok=True)
write_colored_excel(df, file_name=path_)

In [15]:
def export_colored_excel(df, file_path="output.xlsx"):
    from openpyxl import Workbook
    from openpyxl.styles import PatternFill
    import random

    wb = Workbook()
    ws = wb.active
    ws.title = "Data"

    # -----------------------
    # Write header
    # -----------------------
    headers = list(df.columns)
    ws.append(headers)

    # -----------------------
    # Generate colors per column (except time)
    # -----------------------
    value_columns = [col for col in df.columns if col != "time"]

    def random_color():
        return ''.join([format(random.randint(0, 255), '02X') for _ in range(3)])

    color_map = {col: random_color() for col in value_columns}

    # -----------------------
    # Write data + apply colors per column
    # -----------------------
    for _, row in df.iterrows():
        ws.append(list(row))
        current_row = ws.max_row

        for col_idx, col_name in enumerate(headers, start=1):
            if col_name in color_map:
                fill = PatternFill(start_color=color_map[col_name],
                                   end_color=color_map[col_name],
                                   fill_type="solid")
                ws.cell(row=current_row, column=col_idx).fill = fill

    # -----------------------
    # Save file
    # -----------------------
    wb.save(file_path)

In [16]:
export_colored_excel(df, "dev/option_chain_colored.xlsx")

In [17]:
df.columns

Index(['time', 'nifty', 'atm', 'ATM-3_CE', 'ATM-3_PE', 'ATM-2_CE', 'ATM-2_PE',
       'ATM-1_CE', 'ATM-1_PE', 'ATM_CE', 'ATM_PE', 'ATM+1_CE', 'ATM+1_PE',
       'ATM+2_CE', 'ATM+2_PE', 'ATM+3_CE', 'ATM+3_PE'],
      dtype='str')

In [18]:
df

,time,nifty,atm,ATM-3_CE,ATM-3_PE,ATM-2_CE,ATM-2_PE,ATM-1_CE,ATM-1_PE,ATM_CE,ATM_PE,ATM+1_CE,ATM+1_PE,ATM+2_CE,ATM+2_PE,ATM+3_CE,ATM+3_PE
0,2026-04-27 09:10:05.785,23945.45,23950,NIFTY26APR23800CE,NIFTY26APR23800PE,NIFTY26APR23850CE,NIFTY26APR23850PE,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE
1,2026-04-27 09:15:01.784,23980.80,24000,NIFTY26APR23850CE,NIFTY26APR23850PE,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE
2,2026-04-27 09:15:08.281,24027.85,24050,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE
3,2026-04-27 09:15:08.781,24013.75,24000,NIFTY26APR23850CE,NIFTY26APR23850PE,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE
4,2026-04-27 09:15:09.281,24032.80,24050,NIFTY26APR23900CE,NIFTY26APR23900PE,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
651,2026-04-27 14:58:48.780,24124.50,24100,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE,NIFTY26APR24250CE,NIFTY26APR24250PE
652,2026-04-27 14:58:50.280,24125.75,24150,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE,NIFTY26APR24250CE,NIFTY26APR24250PE,NIFTY26APR24300CE,NIFTY26APR24300PE
653,2026-04-27 14:58:50.780,24124.10,24100,NIFTY26APR23950CE,NIFTY26APR23950PE,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE,NIFTY26APR24250CE,NIFTY26APR24250PE
654,2026-04-27 14:58:54.280,24125.15,24150,NIFTY26APR24000CE,NIFTY26APR24000PE,NIFTY26APR24050CE,NIFTY26APR24050PE,NIFTY26APR24100CE,NIFTY26APR24100PE,NIFTY26APR24150CE,NIFTY26APR24150PE,NIFTY26APR24200CE,NIFTY26APR24200PE,NIFTY26APR24250CE,NIFTY26APR24250PE,NIFTY26APR24300CE,NIFTY26APR24300PE
